# 02 · Caixa **ou** recorte — o que muda quando a máquina desenha o contorno

Demo curta, para emendar na 01. A pergunta de palco é: *"por que existe mais de
um tipo de modelo?"*

- **Detecção** responde *onde está* → caixa retangular
- **Segmentação** responde *qual é exatamente o pixel* → contorno recortado

A caixa serve para contar. O contorno serve para **medir**: área ocupada, quanto
de uma prateleira está cheia, quanto de um terreno é telhado.

In [ ]:
# ── 1. instala a biblioteca e monta o Google Drive ──
%pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')

from ultralytics import YOLO
import ultralytics, torch, os, glob
ultralytics.checks()
print("GPU disponivel:", torch.cuda.is_available())

# ── 2. a raiz de tudo, e a conferência de que ela é REAL ──────────────
#
# ARMADILHA que já custou uma sessão: a linha acima cria a variável
# `drive` (minúscula), que é o MÓDULO do Colab. Se algum caminho for
# escrito com `drive` em vez de `DRIVE`, o Python aceita numa boa e
# monta um caminho como
#     <module 'google.colab.drive' from '/usr/local/...'>/04-garrafas
# O código roda, cria pastas, exporta arquivos — tudo no disco
# temporário do Colab, que evapora quando a sessão encerra. Nada disso
# chega ao seu Drive, e não há erro nenhum na tela.
#
# A conferência abaixo transforma esse silêncio num aviso imediato.

DRIVE = "/content/drive/MyDrive/PALESTRA-IA"

if not DRIVE.startswith("/content/drive/"):
    raise SystemExit(
        "DRIVE aponta para fora do Google Drive: " + repr(DRIVE) + "\n"
        "Provavelmente algum caminho usou `drive` (o módulo) em vez de `DRIVE`.")
if not os.path.isdir("/content/drive/MyDrive"):
    raise SystemExit("O Drive não montou. Rode esta célula de novo e autorize o acesso.")

os.makedirs(DRIVE, exist_ok=True)
print("raiz no Drive:", DRIVE)
print("existe de verdade:", os.path.isdir(DRIVE))

In [ ]:
# ── ajuste de PALCO: tudo grande, porque a sala enxerga de 6 a 10 m ──
import matplotlib
matplotlib.rcParams.update({
    "figure.figsize": (16, 9),
    "figure.dpi": 110,
    "font.size": 22,
    "axes.titlesize": 30,
    "axes.labelsize": 24,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 22,
    "axes.grid": True,
    "grid.alpha": .25,
    "axes.facecolor": "#0d1117",
    "figure.facecolor": "#0d1117",
    "text.color": "#e6edf3",
    "axes.labelcolor": "#e6edf3",
    "xtick.color": "#e6edf3",
    "ytick.color": "#e6edf3",
    "axes.edgecolor": "#30363d",
    "axes.titlecolor": "#3fe0a8",
})
VERDE, VERMELHO, CINZA = "#3fe0a8", "#ff5c5c", "#7d8590"
# DRIVE não é redefinido aqui de propósito: quem define é a célula de
# setup, e uma variável de caminho com duas origens é como se perde a
# noção de onde os arquivos foram parar.
print("palco configurado")

In [ ]:
det = YOLO(f"{DRIVE}/00-pesos/yolo11n.pt")
seg = YOLO(f"{DRIVE}/00-pesos/yolo11n-seg.pt")

import glob
entradas = [e for e in sorted(glob.glob(f"{DRIVE}/02-segmentacao/entrada/*"))
            if e.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))]
IMG = entradas[0] if entradas else "https://ultralytics.com/images/bus.jpg"
print("usando:", IMG)

In [ ]:
# ── lado a lado: a mesma cena, dois modelos ──
import matplotlib.pyplot as plt, cv2
rd = det.predict(IMG, conf=.35, verbose=False)[0]
rs = seg.predict(IMG, conf=.35, verbose=False)[0]

fig, axs = plt.subplots(1, 2, figsize=(22, 10))
for ax, r, t in ((axs[0], rd, "DETECÇÃO · onde está"),
                 (axs[1], rs, "SEGMENTAÇÃO · qual pixel é")):
    ax.imshow(cv2.cvtColor(r.plot(line_width=4), cv2.COLOR_BGR2RGB))
    ax.set_title(t, fontsize=30); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# ── o que so a segmentacao entrega: AREA de cada objeto ──
import numpy as np
if rs.masks is None:
    print("nenhuma mascara nesta imagem")
else:
    total_px = rs.orig_shape[0] * rs.orig_shape[1]
    linhas = []
    for m, c in zip(rs.masks.data.cpu().numpy(), rs.boxes.cls):
        linhas.append((seg.names[int(c)], m.sum() / m.size * 100))
    linhas.sort(key=lambda x: -x[1])
    print(f"{'objeto':<16} {'% da imagem':>12}")
    print("-" * 30)
    for nome, pct in linhas[:10]:
        print(f"{nome:<16} {pct:>11.1f}%")
    print("-" * 30)
    print(f"{'ocupacao total':<16} {sum(p for _, p in linhas):>11.1f}%")

> **Fala de palco:** *"a caixa me diz que tem cinco garrafas. O contorno me diz
> que a prateleira está 38% vazia. São perguntas diferentes — e é por isso que
> escolher o modelo é parte do problema, não detalhe técnico."*

---

# 🔴 AO VIVO · cada pessoa é uma instância

Agora a parte que faz a sala entender **segmentação de instâncias** sem uma
única palavra técnica: a câmera abre, e cada pessoa na cena ganha **a própria
cor e o próprio número**.

Não é "achou pessoas". É "achou **estas** pessoas, separadas uma da outra,
recortadas pixel a pixel, e sabe qual é qual entre um quadro e o outro".

> Fala de palco: *"olhem o número em cima de cada um. Ele não muda quando a
> pessoa anda, e não troca de dono quando duas se cruzam. Isso se chama
> rastreamento, e é o que separa uma foto bonita de um sistema que serve para
> contar gente numa loja."*

**Não precisa treinar nada aqui** — pessoa é uma das 80 classes que o modelo
já conhece. O que impressiona não é o reconhecimento: é a separação.

In [ ]:
# ── motor de webcam ao vivo (leia o comentário: é o truque da demo) ──
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import cv2, numpy as np, PIL.Image, io, time

def iniciar_webcam(largura=640, altura=480):
    # cria o video no navegador + a camada de overlay por cima dele
    display(Javascript('''
      var video, div = null, stream, imgElement, labelElement, captureCanvas;
      var pendingResolve = null, shutdown = false;
      var LARG = %d, ALT = %d;

      function removeDom() {
        if (stream) stream.getVideoTracks()[0].stop();
        if (video) video.remove();
        if (div) div.remove();
        video = null; div = null; stream = null;
        imgElement = null; captureCanvas = null; labelElement = null;
      }

      function onAnimationFrame() {
        if (!shutdown) window.requestAnimationFrame(onAnimationFrame);
        if (pendingResolve) {
          var result = "";
          if (!shutdown) {
            captureCanvas.getContext('2d').drawImage(video, 0, 0, LARG, ALT);
            result = captureCanvas.toDataURL('image/jpeg', 0.75);
          }
          var lp = pendingResolve;
          pendingResolve = null;
          lp(result);
        }
      }

      async function criarDom() {
        if (div !== null) return stream;

        div = document.createElement('div');
        div.style.border = '2px solid #3fe0a8';
        div.style.padding = '3px';
        div.style.width = '100%%';
        div.style.maxWidth = '900px';
        div.style.borderRadius = '10px';
        document.body.appendChild(div);

        var parar = document.createElement('div');
        parar.innerHTML = '&#9632; clique aqui para encerrar';
        parar.style.cssText = 'cursor:pointer;background:#3fe0a8;color:#06231a;' +
          'font-weight:700;padding:10px 16px;border-radius:8px;text-align:center;' +
          'font-family:system-ui,sans-serif;font-size:18px';
        div.appendChild(parar);
        parar.onclick = function() { shutdown = true; };

        video = document.createElement('video');
        video.style.display = 'block';
        video.style.width = '100%%';
        video.setAttribute('playsinline', '');
        video.onclick = function() { shutdown = true; };

        stream = await navigator.mediaDevices.getUserMedia(
          {video: {width: LARG, height: ALT}});
        div.appendChild(video);
        video.srcObject = stream;
        await video.play();

        // a camada que recebe o resultado do modelo, por cima do vídeo
        imgElement = document.createElement('img');
        imgElement.style.position = 'absolute';
        imgElement.style.zIndex = 1;
        imgElement.style.pointerEvents = 'none';
        imgElement.onclick = function() { shutdown = true; };
        div.appendChild(imgElement);

        labelElement = document.createElement('div');
        labelElement.style.cssText = 'font-family:system-ui,sans-serif;' +
          'font-size:20px;color:#e6edf3;padding:8px 4px';
        div.appendChild(labelElement);

        captureCanvas = document.createElement('canvas');
        captureCanvas.width = LARG;
        captureCanvas.height = ALT;
        window.requestAnimationFrame(onAnimationFrame);

        google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
        return stream;
      }

      async function quadro(rotulo, overlay) {
        if (shutdown) { removeDom(); shutdown = false; return ''; }
        stream = await criarDom();
        if (rotulo != "") labelElement.innerHTML = rotulo;
        if (overlay != "") {
          var r = video.getClientRects()[0];
          imgElement.style.top = r.top + "px";
          imgElement.style.left = r.left + "px";
          imgElement.style.width = r.width + "px";
          imgElement.style.height = r.height + "px";
          imgElement.src = overlay;
        }
        var result = await new Promise(function(resolve) { pendingResolve = resolve; });
        shutdown = false;
        return {'img': result};
      }
    ''' % (largura, altura)))


def _para_imagem(resposta):
    # base64 do navegador -> imagem BGR do OpenCV
    if not resposta:
        return None
    dados = b64decode(resposta.split(',')[1])
    arr = np.frombuffer(dados, dtype=np.uint8)
    return cv2.imdecode(arr, flags=1)


def _para_overlay(rgba):
    # array RGBA -> data URI PNG, para o navegador sobrepor ao video
    img = PIL.Image.fromarray(rgba, 'RGBA')
    buf = io.BytesIO()
    img.save(buf, format='png')
    return 'data:image/png;base64,' + b64encode(buf.getvalue()).decode('utf-8')


def rodar_ao_vivo(processa, largura=640, altura=480, rotulo_inicial='iniciando…'):
    # Laco principal.
    #
    # `processa(frame_bgr, overlay_rgba)` recebe o quadro e uma tela RGBA
    # transparente do mesmo tamanho, desenha nela, e devolve o texto do
    # painel. Encerre clicando no botão verde (ou no próprio vídeo).
    iniciar_webcam(largura, altura)
    overlay = np.zeros([altura, largura, 4], dtype=np.uint8)
    envio = ''
    rotulo = rotulo_inicial
    n = 0
    t0 = time.time()
    try:
        while True:
            resposta = eval_js('quadro("{}", "{}")'.format(rotulo, envio))
            if not resposta:
                break
            frame = _para_imagem(resposta['img'])
            if frame is None:
                break

            overlay[:] = 0
            texto = processa(frame, overlay)

            n += 1
            fps = n / max(1e-6, time.time() - t0)
            rotulo = '{} &nbsp;·&nbsp; {:.1f} quadros/s'.format(texto, fps)
            envio = _para_overlay(overlay)
    except Exception as e:
        print('encerrado:', type(e).__name__, e)
    print('fim · {} quadros processados'.format(n))

In [ ]:
# ── cores por instância ──────────────────────────────────────────────
# A cor precisa ser SEMPRE a mesma para o mesmo id, senão a plateia acha
# que o sistema se perdeu. Por isso ela é derivada do próprio número,
# não sorteada.
import cv2, numpy as np

def cor_do_id(i):
    matiz = int((i * 47) % 180)                # espalha bem no circulo de cor
    hsv = np.uint8([[[matiz, 210, 255]]])
    b, g, r = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)[0][0]
    return int(r), int(g), int(b)              # devolve em RGB

def pintar_mascara(overlay, mascara, cor_rgb, alpha=110):
    """Pinta a máscara no overlay RGBA e contorna a borda."""
    m = mascara.astype(bool)
    if not m.any():
        return
    overlay[m] = (cor_rgb[0], cor_rgb[1], cor_rgb[2], alpha)
    contornos, _ = cv2.findContours(mascara.astype(np.uint8),
                                    cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(overlay, contornos, -1, (*cor_rgb, 255), 3)

print("paleta pronta")

In [ ]:
# ── o laço ao vivo: instâncias de pessoa, com id estável ─────────────
segmentador = YOLO(f"{DRIVE}/00-pesos/yolo11n-seg.pt")
CLASSE_PESSOA = 0

vistos = set()          # todos os ids que já apareceram na sessão

def processa_instancias(frame, overlay):
    h, w = frame.shape[:2]
    r = segmentador.track(frame, persist=True, conf=.45, classes=[CLASSE_PESSOA],
                          verbose=False, imgsz=480)[0]

    na_cena = 0
    if r.masks is not None and r.boxes is not None and len(r.boxes):
        na_cena = len(r.boxes)
        ids = (r.boxes.id.cpu().numpy().astype(int) if r.boxes.id is not None
               else np.arange(na_cena))
        mascaras = r.masks.data.cpu().numpy()

        for pid, mask, caixa in zip(ids, mascaras, r.boxes.xyxy.cpu().numpy()):
            vistos.add(int(pid))
            # a máscara sai na resolução do modelo: precisa voltar ao quadro
            m = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
            cor = cor_do_id(int(pid))
            pintar_mascara(overlay, m, cor)

            x1, y1 = int(caixa[0]), int(caixa[1])
            cv2.rectangle(overlay, (x1, max(0, y1 - 34)), (x1 + 92, y1), (*cor, 235), -1)
            cv2.putText(overlay, "#{}".format(pid), (x1 + 8, max(22, y1 - 8)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (10, 15, 20, 255), 2)

    cv2.rectangle(overlay, (0, 0), (w, 54), (13, 17, 23, 210), -1)
    cv2.putText(overlay, "NA CENA {}".format(na_cena), (14, 38),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (230, 237, 243, 255), 2)
    cv2.putText(overlay, "JA VISTAS {}".format(len(vistos)), (int(w * .45), 38),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (63, 224, 168, 255), 2)

    return ("<b>{}</b> pessoas na cena &nbsp;·&nbsp; "
            "<b>{}</b> instâncias diferentes desde que a câmera abriu"
            .format(na_cena, len(vistos)))

vistos.clear()
rodar_ao_vivo(processa_instancias, largura=640, altura=480,
              rotulo_inicial='autorize a câmera…')

### O que dizer enquanto roda

1. **Peça para duas pessoas se cruzarem** na frente da câmera. Os números não
   trocam de dono — é a prova de que ele rastreia, não só detecta.
2. **Peça para alguém sair e voltar.** Vai ganhar um número novo: o sistema
   não sabe *quem* é a pessoa, sabe que é *outra instância*. Boa deixa para
   falar de privacidade — ele conta pessoas sem identificar ninguém.
3. O contador **"já vistas"** é o mesmo mecanismo de um contador de fluxo de
   loja: quantas pessoas passaram, não quantas estão paradas ali agora.